# U8-04 · Una red de principio a fin

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 8, apartado 8.10*

Ejecuta cada celda con **Mayús + Intro**. Solo se usan NumPy y scikit-learn; no hace falta tarjeta gráfica. La búsqueda prueba 15 combinaciones con validación cruzada de 5 bloques: en el navegador tarda unos minutos.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

X, y = load_digits(return_X_y=True)
X = X.astype(np.float32)                     # la mitad de memoria
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)

modelo = make_pipeline(StandardScaler(),
                       MLPClassifier(max_iter=500, random_state=0))
rejilla = {"mlpclassifier__hidden_layer_sizes":
               [(16,), (32,), (64,), (128,), (64, 64)],
           "mlpclassifier__alpha": [1e-4, 1e-2, 1.0]}
busqueda = GridSearchCV(modelo, rejilla, cv=5).fit(X_tr, y_tr)

res = pd.DataFrame(busqueda.cv_results_)
col = "param_mlpclassifier__hidden_layer_sizes"
res["capas"] = res[col].astype(str)
res["alpha"] = res["param_mlpclassifier__alpha"]
print(res.pivot(index="capas", columns="alpha",
                values="mean_test_score").round(3))

def parametros(capas, entradas=64, salidas=10):
    tam = [entradas, *capas, salidas]
    return sum((a + 1) * b for a, b in zip(tam[:-1], tam[1:]))

res["parametros"] = res[col].map(parametros)
# el mejor en validación cruzada; si empatan, el más pequeño
orden = res.sort_values(["mean_test_score", "parametros"],
                        ascending=[False, True])
elegido = orden.iloc[0]
print("elegido:", elegido["capas"], elegido["alpha"],
      elegido["parametros"], "parámetros")

final = modelo.set_params(**elegido["params"]).fit(X_tr, y_tr)
print("exactitud en prueba:", round(final.score(X_te, y_te), 4))

from sklearn.metrics import confusion_matrix
import joblib, os
M = confusion_matrix(y_te, final.predict(X_te))
np.fill_diagonal(M, 0)
peores = np.argsort(M, axis=None)[::-1][:3]
print("errores:", M.sum(), "· más frecuentes (real, predicho):",
      [divmod(int(i), 10) for i in peores])
joblib.dump(final, "digitos.joblib")
print(os.path.getsize("digitos.joblib") // 1024, "KB")

**Ejercicio.** Añade `(8,)` a la rejilla y `alpha=3.0`. ¿Cambia el modelo elegido? Carga después el modelo guardado con `joblib.load("digitos.joblib")` y predice las cinco primeras imágenes de prueba.

> Kingma, D. P. y Ba, J. (2015). Adam: A Method for Stochastic Optimization. *ICLR*. https://doi.org/10.48550/arXiv.1412.6980